# Applied Data Visualization and EDA with the UCI Iris Dataset

This notebook turns the repository's two companion articles into an executable demonstration:

1. **What are the important principles of data visualization?**
2. **Exploratory Data Analysis is a significant part of Data Science**

The public Iris dataset originates from the UCI Machine Learning Repository and is loaded through scikit-learn, so the notebook is reproducible without a fragile network download.

## Learning goals

- inspect data quality before plotting;
- choose charts from analytical questions;
- compare distributions and categories;
- examine relationships and correlations;
- reduce visual noise and use honest scales;
- add titles, units, annotations, and direct labels;
- move from exploration to a concise visual narrative.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.datasets import load_iris

from dataviz_reference import apply_accessible_theme

ARTIFACTS = Path("../artifacts")
ARTIFACTS.mkdir(exist_ok=True)

apply_accessible_theme()
sns.set_context("notebook")


## 1. Load a governed external dataset

`load_iris(as_frame=True)` provides the original public Iris measurements as a pandas DataFrame.
The target is translated into readable species names before analysis.


In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.rename(
    columns={
        "sepal length (cm)": "sepal_length_cm",
        "sepal width (cm)": "sepal_width_cm",
        "petal length (cm)": "petal_length_cm",
        "petal width (cm)": "petal_width_cm",
    }
)
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df = df.drop(columns="target")
df.head()


## 2. Build a relationship with the data before charting

EDA starts with structure, data types, missing values, duplicates, and summary statistics.
This prevents a polished chart from hiding a basic data-quality problem.


In [ ]:
quality_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "unique_values": df.nunique(),
    }
)
display(quality_summary)
display(df.describe(include="all").T)
print(f"Rows: {len(df):,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")


## 3. Analytical question: how are species represented?

A sorted bar chart is more readable than an unordered category chart.
The axis starts at zero because bar length encodes magnitude directly.


In [ ]:
species_counts = df["species"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(species_counts.index, species_counts.values)
ax.set(
    title="The Iris dataset is balanced across species",
    xlabel="Number of observations",
    ylabel="",
    xlim=(0, species_counts.max() * 1.18),
)
ax.bar_label(bars, padding=4)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(ARTIFACTS / "iris_species_counts.png", dpi=160, bbox_inches="tight")
plt.show()


## 4. Analytical question: how do feature distributions differ?

Small multiples keep the same visual grammar while avoiding one overcrowded chart.
Density is used for shape comparison; units are explicit in the axis labels.


In [ ]:
features = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
]
long_df = df.melt(
    id_vars="species",
    value_vars=features,
    var_name="feature",
    value_name="measurement_cm",
)
grid = sns.displot(
    data=long_df,
    x="measurement_cm",
    hue="species",
    col="feature",
    col_wrap=2,
    kind="kde",
    fill=False,
    common_norm=False,
    height=3.2,
    facet_kws={"sharex": False, "sharey": False},
)
grid.set_axis_labels("Measurement (cm)", "Density")
grid.set_titles("{col_name}")
grid.figure.suptitle(
    "Petal measurements separate species more clearly than sepal measurements",
    y=1.03,
)
grid.figure.savefig(
    ARTIFACTS / "iris_feature_distributions.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 5. Analytical question: which measurements best separate species?

A scatter plot is appropriate because both axes are quantitative.
Colour identifies species, while transparency limits overlap.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.scatterplot(
    data=df,
    x="petal_length_cm",
    y="petal_width_cm",
    hue="species",
    style="species",
    s=75,
    alpha=0.8,
    ax=ax,
)
ax.set(
    title="Petal length and width reveal three distinct species groups",
    xlabel="Petal length (cm)",
    ylabel="Petal width (cm)",
)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.2)
ax.legend(title="Species", frameon=False)
fig.tight_layout()
fig.savefig(ARTIFACTS / "iris_petal_relationship.png", dpi=160, bbox_inches="tight")
plt.show()


## 6. Analytical question: which numeric variables move together?

A correlation heatmap summarizes linear association, but it does not prove causation.
Annotations make exact values available without forcing readers to infer them from colour alone.


In [ ]:
correlation = df[features].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Petal length and width have the strongest linear relationship")
fig.tight_layout()
fig.savefig(ARTIFACTS / "iris_correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Add interaction only when it improves understanding

Hover details allow a reader to inspect individual flowers without adding permanent labels to every point.
The chart remains interpretable before any interaction.


In [ ]:
interactive = px.scatter(
    df,
    x="petal_length_cm",
    y="petal_width_cm",
    color="species",
    symbol="species",
    hover_data=["sepal_length_cm", "sepal_width_cm"],
    labels={
        "petal_length_cm": "Petal length (cm)",
        "petal_width_cm": "Petal width (cm)",
        "species": "Species",
    },
    title="Interactive Iris species comparison",
)
interactive.update_layout(
    template="plotly_white",
    legend_title_text="Species",
    hovermode="closest",
)
interactive.write_html(
    ARTIFACTS / "iris_interactive_scatter.html",
    include_plotlyjs="cdn",
)
interactive.show()


## 8. Move from exploration to a narrative conclusion

The strongest visual story is not “we drew several charts.” It is:

- the dataset is balanced and has no missing values;
- petal length and petal width vary substantially by species;
- those two measurements are strongly correlated;
- they separate the three species much more clearly than sepal measurements;
- an interactive scatter plot is useful for inspecting individual observations, while the static version is better for a concise report.

This sequence follows the articles' shared principle: **use EDA to discover evidence, then refine the visual encoding until the message is clear.**


In [ ]:
species_profile = (
    df.groupby("species")[features]
    .mean()
    .round(2)
    .sort_values("petal_length_cm")
)
species_profile


## Reproducibility and citation

- Dataset: Fisher's Iris dataset, distributed through scikit-learn and originally hosted by the UCI Machine Learning Repository.
- The notebook does not require a live data download.
- Generated files are written to `artifacts/`.
- Cite the relevant Medium article for the conceptual principles and this repository commit for the implementation.
